In [2]:
from FeatureCloud.app.engine.app import AppState, app_state, Role
import time
import bios
import pandas as pd
import numpy as np
import sklearn as sk
from sklearn.linear_model import LinearRegression

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from torch import nn
from sklearn.cluster import KMeans

In [34]:
df = pd.read_csv('data/real_dataset_normalized.tsv', sep='\t', index_col=0)
df


C:\Users\david\AppData\Local\Temp\ipykernel_13768\658711780.py:1: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/real_dataset_normalized.tsv', sep='\t', index_col=0)


,01CPTAC_1,01CPTAC_2,01CPTAC_3,01CPTAC_4,01CPTAC_5,01CPTAC_6,01CPTAC_7,01CPTAC_8,01CPTAC_9,01CPTAC_10,...,13CPTAC_1,13CPTAC_2,13CPTAC_3,13CPTAC_4,13CPTAC_5,13CPTAC_6,13CPTAC_7,13CPTAC_8,13CPTAC_9,13CPTAC_10
0,0.935603,-0.086369,0.786048,1.069258,0.860449,-0.130177,-0.311762,0.131637,0.537098,1.431399,...,0.632951,1.099141,1.107433,0.974463,1.019426,1.630872,1.682706,0.816556,0.104716,-0.496306
1,1.551572,-0.197678,0.370067,-0.161171,0.359997,-0.262972,-0.413216,0.394467,-0.055829,0.447476,...,1.002573,-0.051359,0.999509,-0.185467,-0.015808,0.012965,-0.818991,-0.60685,-0.62555,0.192115
2,-0.473628,-2.233779,-1.483126,-0.341969,-0.6778,-1.742391,-2.344356,-1.39933,-0.273224,-0.258776,...,1.126813,1.173089,1.45008,0.16036,1.093243,1.152555,0.800632,0.648046,-0.505979,-0.479573
3,0.668724,0.438768,0.518318,0.46211,0.445675,0.481358,0.208015,0.381343,0.464918,0.480309,...,0.736181,0.532446,0.898038,0.501932,0.689904,0.679189,0.620348,0.570685,0.405587,0.540742
4,-1.199512,-2.270995,-2.151234,-1.813642,-2.031302,-1.815763,-1.708907,-2.092016,-2.260684,-1.897957,...,1.491828,0.755135,1.422832,1.76043,1.342536,0.76432,0.765718,1.032165,1.650714,1.483676
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10560,-0.285321951822366,-0.285321951822366,-0.285321951822366,-0.285321951822366,-0.285321951822366,-0.285321951822366,-0.285321951822366,-0.285321951822366,-0.285321951822366,-0.285321951822366,...,-0.285321951822366,-0.285321951822366,-0.285321951822366,-0.285321951822366,-0.285321951822366,-0.285321951822366,-0.285321951822366,-0.285321951822366,-0.285321951822366,-0.285321951822366
10561,1.2446928958307906,-0.18542103396927265,-0.18807113964624592,-0.4326992285610106,-0.5489280354050711,-0.7635051859790253,-1.0006968162223688,-0.19832776679947425,-1.1383480937268962,-0.3398389956764241,...,0.6842244680423223,-1.0940010252406491,1.169143187382292,1.090343890626097,0.19966072783619965,-1.000406893836414,-1.4371923372927784,-0.5007327677267152,-0.9074357528064427,-0.5542534145726915
10562,0.26414242807278165,0.15250367712623575,0.21817441032275073,0.16396493284288707,0.20936256336208922,0.24100416513910672,0.07190893691057308,0.23307358983243565,0.2582638694975213,0.28615078226944496,...,0.5883878164402481,0.33144023011044876,0.3802116564300159,0.5002482193440194,0.33682699641432484,0.3778016385904391,0.25971580883452655,0.29463495531353984,0.18195309388126177,0.1870420162907245
10563,-0.4763250630956321,-1.209986253176779,-0.25758305025144523,-0.6455604351342914,-0.4960141905367187,-1.18325375153811,-0.9426291498872269,-1.0690614771742197,-0.4275900430396228,-0.29720728346551745,...,0.9601407764177715,0.888155089617438,0.8624895414050053,1.1359257532098395,1.1092011180864818,0.7484195407261035,0.9493487260711219,1.1266863464518508,0.7768642993389901,0.7905239938237515


In [45]:
sample_data = df
sample_data

,01CPTAC_1,01CPTAC_2,01CPTAC_3,01CPTAC_4,01CPTAC_5,01CPTAC_6,01CPTAC_7,01CPTAC_8,01CPTAC_9,01CPTAC_10,...,13CPTAC_1,13CPTAC_2,13CPTAC_3,13CPTAC_4,13CPTAC_5,13CPTAC_6,13CPTAC_7,13CPTAC_8,13CPTAC_9,13CPTAC_10
0,0.935603,-0.086369,0.786048,1.069258,0.860449,-0.130177,-0.311762,0.131637,0.537098,1.431399,...,0.632951,1.099141,1.107433,0.974463,1.019426,1.630872,1.682706,0.816556,0.104716,-0.496306
1,1.551572,-0.197678,0.370067,-0.161171,0.359997,-0.262972,-0.413216,0.394467,-0.055829,0.447476,...,1.002573,-0.051359,0.999509,-0.185467,-0.015808,0.012965,-0.818991,-0.60685,-0.62555,0.192115
2,-0.473628,-2.233779,-1.483126,-0.341969,-0.6778,-1.742391,-2.344356,-1.39933,-0.273224,-0.258776,...,1.126813,1.173089,1.45008,0.16036,1.093243,1.152555,0.800632,0.648046,-0.505979,-0.479573
3,0.668724,0.438768,0.518318,0.46211,0.445675,0.481358,0.208015,0.381343,0.464918,0.480309,...,0.736181,0.532446,0.898038,0.501932,0.689904,0.679189,0.620348,0.570685,0.405587,0.540742
4,-1.199512,-2.270995,-2.151234,-1.813642,-2.031302,-1.815763,-1.708907,-2.092016,-2.260684,-1.897957,...,1.491828,0.755135,1.422832,1.76043,1.342536,0.76432,0.765718,1.032165,1.650714,1.483676
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10560,-0.285321951822366,-0.285321951822366,-0.285321951822366,-0.285321951822366,-0.285321951822366,-0.285321951822366,-0.285321951822366,-0.285321951822366,-0.285321951822366,-0.285321951822366,...,-0.285321951822366,-0.285321951822366,-0.285321951822366,-0.285321951822366,-0.285321951822366,-0.285321951822366,-0.285321951822366,-0.285321951822366,-0.285321951822366,-0.285321951822366
10561,1.2446928958307906,-0.18542103396927265,-0.18807113964624592,-0.4326992285610106,-0.5489280354050711,-0.7635051859790253,-1.0006968162223688,-0.19832776679947425,-1.1383480937268962,-0.3398389956764241,...,0.6842244680423223,-1.0940010252406491,1.169143187382292,1.090343890626097,0.19966072783619965,-1.000406893836414,-1.4371923372927784,-0.5007327677267152,-0.9074357528064427,-0.5542534145726915
10562,0.26414242807278165,0.15250367712623575,0.21817441032275073,0.16396493284288707,0.20936256336208922,0.24100416513910672,0.07190893691057308,0.23307358983243565,0.2582638694975213,0.28615078226944496,...,0.5883878164402481,0.33144023011044876,0.3802116564300159,0.5002482193440194,0.33682699641432484,0.3778016385904391,0.25971580883452655,0.29463495531353984,0.18195309388126177,0.1870420162907245
10563,-0.4763250630956321,-1.209986253176779,-0.25758305025144523,-0.6455604351342914,-0.4960141905367187,-1.18325375153811,-0.9426291498872269,-1.0690614771742197,-0.4275900430396228,-0.29720728346551745,...,0.9601407764177715,0.888155089617438,0.8624895414050053,1.1359257532098395,1.1092011180864818,0.7484195407261035,0.9493487260711219,1.1266863464518508,0.7768642993389901,0.7905239938237515


In [36]:
conditions = df.T['Conditions']
sample_data = df.T.drop(columns=['Conditions'])
sample_data

,0,1,2,3,4,5,6,7,8,9,...,10554,10555,10556,10557,10558,10559,10560,10561,10562,10563
01CPTAC_1,0.935603,1.551572,-0.473628,0.668724,-1.199512,0.229795,-1.126011,-1.805717,0.184352,0.867927,...,0.4569293088272782,0.473249894256092,-1.2434474311668966,0.3569417455163055,-0.4940733079086548,-0.017790091556662037,-0.285321951822366,1.2446928958307906,0.26414242807278165,-0.4763250630956321
01CPTAC_2,-0.086369,-0.197678,-2.233779,0.438768,-2.270995,-1.25026,-2.324319,-2.701704,-1.347024,0.704373,...,0.15534962214738657,-0.9157017586739586,-1.2434474311668966,-0.14119030119410508,-0.4940733079086548,-0.469880029527231,-0.285321951822366,-0.18542103396927265,0.15250367712623575,-1.209986253176779
01CPTAC_3,0.786048,0.370067,-1.483126,0.518318,-2.151234,-1.439751,-1.43155,-2.866629,0.272951,0.83598,...,0.058572598007062424,-0.5302627370605081,-1.2434474311668966,0.12440478267380878,-0.4940733079086548,0.010122899250728354,-0.285321951822366,-0.18807113964624592,0.21817441032275073,-0.25758305025144523
01CPTAC_4,1.069258,-0.161171,-0.341969,0.46211,-1.813642,-1.612057,-1.093427,-2.101206,-0.644707,0.662508,...,0.22178725033452124,-0.4259283053455973,-1.2434474311668966,0.09619956475038913,-0.4940733079086548,-0.3021596110209463,-0.285321951822366,-0.4326992285610106,0.16396493284288707,-0.6455604351342914
01CPTAC_5,0.860449,0.359997,-0.6778,0.445675,-2.031302,0.704869,-1.200141,-1.963744,0.006111,0.873364,...,0.20973429622871764,-0.2721946701818993,-1.2434474311668966,0.05789572702811481,-0.4940733079086548,-0.08998775580535515,-0.285321951822366,-0.5489280354050711,0.20936256336208922,-0.4960141905367187
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13CPTAC_6,1.630872,0.012965,1.152555,0.679189,0.76432,0.529369,1.784231,1.523567,-0.02789,1.093415,...,0.7630837063704837,1.4719631637354453,0.7829031726029733,0.9030533433063662,-0.4940733079086548,0.6362748187825336,-0.285321951822366,-1.000406893836414,0.3778016385904391,0.7484195407261035
13CPTAC_7,1.682706,-0.818991,0.800632,0.620348,0.765718,0.902489,1.930407,0.653648,0.126765,0.861804,...,0.7019048699407728,2.0225597292143154,0.7272292170240727,0.7706392334040808,-0.4940733079086548,0.9343670154558037,-0.285321951822366,-1.4371923372927784,0.25971580883452655,0.9493487260711219
13CPTAC_8,0.816556,-0.60685,0.648046,0.570685,1.032165,0.435475,-0.049591,0.64221,0.388394,0.849829,...,0.6650138580661155,0.795976052436256,0.6031663603496312,0.8284936564485966,-0.4940733079086548,0.6650859043127806,-0.285321951822366,-0.5007327677267152,0.29463495531353984,1.1266863464518508
13CPTAC_9,0.104716,-0.62555,-0.505979,0.405587,1.650714,1.468639,-0.265719,0.338308,0.110354,0.889028,...,1.3912418278315781,0.14234462709333126,0.6625666804635553,0.9070699608688025,-0.4940733079086548,0.8080970000571848,-0.285321951822366,-0.9074357528064427,0.18195309388126177,0.7768642993389901


In [37]:

part1 = df.iloc[:, :40].T
part2 = df.iloc[:, 40:80].T
part3 = df.iloc[:, 80:130].T

part1_conditions = part1.iloc[:, -1]
part2_conditions = part2.iloc[:, -1]
part3_conditions = part3.iloc[:, -1]

part1.drop(part1.columns[len(part1.columns)-1], axis=1, inplace=True)
part2.drop(part2.columns[len(part2.columns)-1], axis=1, inplace=True)
part3.drop(part3.columns[len(part3.columns)-1], axis=1, inplace=True)

In [38]:
export1 = part1.to_csv('data/client1/realdata.csv', sep=';')
export2 = part2.to_csv('data/client2/realdata.csv', sep=';')
export3 = part3.to_csv('data/client3/realdata.csv', sep=';')
export_meta1 = part1_conditions.to_csv('data/client1/realmetadata.csv', sep=';')
export_meta2 = part2_conditions.to_csv('data/client2/realmetadata.csv', sep=';')
export_meta3 = part3_conditions.to_csv('data/client3/realmetadata.csv', sep=';')

In [3]:
df1 = pd.read_csv('data/client1/realmetadata.csv', sep=';', index_col=0)
df1


,Conditions
01CPTAC_1,Not Reported
01CPTAC_2,Primary Tumor
01CPTAC_3,Primary Tumor
01CPTAC_4,Primary Tumor
01CPTAC_5,Primary Tumor
01CPTAC_6,Primary Tumor
01CPTAC_7,Solid Tissue Normal
01CPTAC_8,Primary Tumor
01CPTAC_9,Primary Tumor
01CPTAC_10,Primary Tumor


In [49]:
sample_data_normalized = df1.reset_index(drop=True).to_numpy(dtype=np.float32)
sample_data_normalized = np.nan_to_num(sample_data_normalized)
proteomics_tensor = torch.tensor(sample_data_normalized, dtype=torch.float32)

In [50]:
import os.path as op
import time
import bios
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from Autoencoder_classes import Encoder, Decoder, GOAE, ClusteringLayer, extract_latent_space, orthogonality_loss
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

In [51]:
input_dim = proteomics_tensor.shape[1]

latent_dim = 10
hidden_dim_1 = 500
hidden_dim_2 = 2000
hidden_dim_3 = 500
n_clusters = 2

# Initialize the autoencoder
autoencoder = GOAE(input_dim, latent_dim, hidden_dim_1=hidden_dim_1, hidden_dim_2=hidden_dim_2, hidden_dim_3=hidden_dim_3)
#clustering_layer = ClusteringLayer(n_clusters, latent_dim)

# Initialize k-means clustering


with torch.no_grad():
    latent_representations = extract_latent_space(autoencoder, proteomics_tensor)
kmeans = KMeans(n_clusters=n_clusters, random_state=22)
kmeans.fit(latent_representations)

KMeans(n_clusters=2, random_state=22)

In [15]:
df = pd.read_csv('data/real_dataset_raw.tsv', sep = '\t')
print(df.isnull().sum().sum())
#print(df[df.isnull().any(axis=1)])  # Rows with NaNs
non_numeric_cols = df.select_dtypes(exclude=['number']).columns
print("Non-numeric columns:", non_numeric_cols)
df

0
Non-numeric columns: Index(['Unnamed: 0', '01CPTAC_1', '01CPTAC_2', '01CPTAC_3', '01CPTAC_4',
       '01CPTAC_5', '01CPTAC_6', '01CPTAC_7', '01CPTAC_8', '01CPTAC_9',
       ...
       '13CPTAC_1', '13CPTAC_2', '13CPTAC_3', '13CPTAC_4', '13CPTAC_5',
       '13CPTAC_6', '13CPTAC_7', '13CPTAC_8', '13CPTAC_9', '13CPTAC_10'],
      dtype='object', length=131)


C:\Users\david\AppData\Local\Temp\ipykernel_13768\2530032057.py:1: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/real_dataset_raw.tsv', sep = '\t')


,Unnamed: 0,01CPTAC_1,01CPTAC_2,01CPTAC_3,01CPTAC_4,01CPTAC_5,01CPTAC_6,01CPTAC_7,01CPTAC_8,01CPTAC_9,...,13CPTAC_1,13CPTAC_2,13CPTAC_3,13CPTAC_4,13CPTAC_5,13CPTAC_6,13CPTAC_7,13CPTAC_8,13CPTAC_9,13CPTAC_10
0,0,10.228393,9.627207,10.140416,10.307017,10.184183,9.601436,9.494617,9.755451,9.993968,...,10.050355,10.324596,10.329474,10.251253,10.277703,10.637393,10.667885,10.158362,9.739615,9.386057
1,1,12.556557,11.253857,11.676667,11.281044,11.669168,11.205231,11.093341,11.694839,11.359495,...,12.147707,11.362823,12.145425,11.26295,11.389299,11.410727,10.791152,10.949138,10.935212,11.544143
2,2,12.771105,11.692084,12.152255,12.851816,12.645942,11.993319,11.624297,12.203625,12.893959,...,13.75222,13.780589,13.950392,13.159757,13.731641,13.768001,13.552262,13.458723,12.751274,12.767461
3,3,9.891162,9.119551,9.386476,9.197873,9.142725,9.262458,8.345265,8.926863,9.207296,...,10.11751,9.433884,10.660618,9.331495,9.962228,9.926276,9.728837,9.562194,9.008212,9.461721
4,4,10.877745,10.077525,10.166967,10.419092,10.256536,10.417508,10.497312,10.211193,10.085226,...,12.887732,12.337544,12.836203,13.088333,12.776235,12.344403,12.345448,12.54444,13.006393,12.881644
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10560,10560,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10561,10561,11.765790912021632,10.634556526094629,10.632460266065506,10.438957007214878,10.34701886145075,10.177286194629282,9.98966524877718,10.62434717166261,9.88078171500049,...,11.322454794157652,9.915860688599269,11.706030338921243,11.643699309050122,10.939160041430364,9.989894580276866,9.64439293693626,10.38514179900184,10.06343568035319,10.342806433400801
10562,10562,7.976286295611792,7.627349629556931,7.83260924151678,7.663172785268207,7.8050670442584895,7.903965634032166,7.375443730270433,7.8791779282942915,7.9579123473808435,...,8.989743574928548,8.186631453334735,8.339070805749548,8.71425556653169,8.203468261870789,8.331538084513019,7.96245051058379,8.071593352567954,7.719396353589872,7.735302225021737
10563,10563,7.317743475817886,6.371047585502042,7.600002054298918,7.099366882683184,7.292337176173877,6.405542458314369,6.71603771766778,6.552892976929313,7.380629783319556,...,9.171318838985215,9.07843052108482,9.04531241514722,9.398146859000708,9.363662136919832,8.898119675077247,9.157393078907699,9.386224586806131,8.934823996423273,8.952450084332138


In [17]:
#df = pd.read_csv('data/client1/allData.csv', sep = ';').apply(pd.to_numeric, errors='coerce')
df = pd.read_csv('data/real_dataset_raw.tsv', sep = '\t', index_col=0)
conditions = df.T['Conditions']
sample_data = df.T.drop(columns=['Conditions'])

#normalisation skipped for now discussion in meeting
sample_data = sample_data.reset_index(drop=True).to_numpy(dtype=np.float32)
sample_data_normalized = (sample_data - sample_data.mean(axis=0))/sample_data.std(axis=0)
sample_data_normalized = np.nan_to_num(sample_data_normalized)

proteomics_tensor = torch.tensor(sample_data_normalized, dtype=torch.float32)

input_dim = proteomics_tensor.shape[1]

latent_dim = 10
hidden_dim_1 = 500
hidden_dim_2 = 2000
hidden_dim_3 = 500
n_clusters = 2

# Initialize the autoencoder
autoencoder = GOAE(input_dim, latent_dim, hidden_dim_1=hidden_dim_1, hidden_dim_2=hidden_dim_2, hidden_dim_3=hidden_dim_3)
clustering_layer = ClusteringLayer(n_clusters, latent_dim)

# Initialize optimizers
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=1e-3)
optimizer_dec = torch.optim.Adam(
    list(autoencoder.parameters()) + list(clustering_layer.parameters()), lr=1e-3
)

with torch.no_grad():
    latent_representations = extract_latent_space(autoencoder, proteomics_tensor)
kmeans = KMeans(n_clusters=n_clusters, random_state=22)
kmeans.fit(latent_representations)
initial_cluster_centers = torch.tensor(kmeans.cluster_centers_, dtype=torch.float32)
clustering_layer.cluster_centers.data = initial_cluster_centers
clustering_layer = latent_representations

C:\Users\david\AppData\Local\Temp\ipykernel_13768\1965521559.py:2: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/real_dataset_raw.tsv', sep = '\t', index_col=0)
C:\Users\david\AppData\Local\Temp\ipykernel_13768\1965521559.py:8: RuntimeWarning: invalid value encountered in divide
  sample_data_normalized = (sample_data - sample_data.mean(axis=0))/sample_data.std(axis=0)


In [ ]:
df_norm = pd.DataFrame(data=sample_data_normalized[1:,1:],index=sample_data_normalized[1:,0],columns=sample_data_normalized[0,1:])  
df_norm

,1.557573,-0.475462,0.671311,-1.204153,0.230684,-1.130367,-1.812702,0.185066,0.871285,-0.130649,...,0.458697,0.475081,-1.248258,0.358322,-0.495985,-0.017859,-0.286426,1.249508,0.265164,-0.478168
-0.086702,-0.198443,-2.242422,0.440466,-2.279781,-1.255096,-2.333311,-2.712155,-1.352234,0.707098,-1.652962,...,0.155951,-0.919244,-1.248258,-0.141737,-0.495985,-0.471698,-0.286426,-0.186138,0.153094,-1.214668
0.789089,0.371498,-1.488864,0.520323,-2.159557,-1.445320,-1.437088,-2.877717,0.274008,0.839214,-1.014456,...,0.058799,-0.532314,-1.248258,0.124886,-0.495985,0.010162,-0.286426,-0.188798,0.219018,-0.258580
1.073395,-0.161795,-0.343292,0.463898,-1.820659,-1.618293,-1.097656,-2.109334,-0.647201,0.665070,-1.016561,...,0.222645,-0.427576,-1.248258,0.096572,-0.495985,-0.303329,-0.286426,-0.434373,0.164599,-0.648058
0.863779,0.361390,-0.680422,0.447399,-2.039162,0.707595,-1.204784,-1.971339,0.006136,0.876743,-0.286292,...,0.210546,-0.273247,-1.248258,0.058120,-0.495985,-0.090336,-0.286426,-0.551051,0.210173,-0.497934
-0.130681,-0.263990,-1.749132,0.483220,-1.822788,-1.432085,-2.586522,-2.996597,-0.824448,0.935383,-1.537167,...,0.667498,-1.121810,-1.248258,0.090673,-0.495985,-0.571541,-0.286426,-0.766459,0.241937,-1.187832
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1.637182,0.013015,1.157013,0.681817,0.767276,0.531416,1.791133,1.529462,-0.027996,1.097645,0.955752,...,0.766035,1.477658,0.785932,0.906547,-0.495985,0.638736,-0.286426,-1.004276,0.379263,0.751314
1.689216,-0.822161,0.803729,0.622748,0.768679,0.905981,1.937875,0.656178,0.127256,0.865138,0.375678,...,0.704620,2.030385,0.730043,0.773620,-0.495985,0.937981,-0.286426,-1.442752,0.260721,0.953021
0.819716,-0.609197,0.650553,0.572893,1.036157,0.437160,-0.049783,0.644695,0.389897,0.853117,0.295788,...,0.667586,0.799055,0.605500,0.831699,-0.495985,0.667659,-0.286426,-0.502670,0.295775,1.131045
0.105122,-0.627970,-0.507937,0.407156,1.657099,1.474321,-0.266746,0.339617,0.110782,0.892467,-0.548910,...,1.396624,0.142895,0.665130,0.910579,-0.495985,0.811223,-0.286426,-0.910946,0.182657,0.779869
